# XMCA-Net-Lite: 06 - Utilization Consistency Scoring (S_U)
### Purpose
Compute Utilization Consistency Score (S_U) by verifying whether answer claims are entailed by retrieved passages using DeBERTa-v3-base MNLI (with LoRA).
Also computes text-only S_U for ablation #5.
Tracks and saves:
- `checkpoint-last`
- `checkpoint-best-auprc`
- `checkpoint-best-ece`


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.utilization.nli import DeBERTaNLIEngine
from src.utilization.scorer import UtilizationScorer
from src.utils.checkpoints import ShardManager, ModelCheckpointTracker
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Load perception and retrieval tables to pair claims and evidence
p_dir = Path(config["paths"]["perception_dir"])
r_dir = Path(config["paths"]["retrieval_dir"])

p_df = ShardManager(p_dir).load_all_shards("perception_scores")
r_df = ShardManager(r_dir).load_all_shards("retrieval_scores")

merged = p_df.merge(r_df[["id", "retrieved_passages"]], on="id")
print(f"Merged records for utilization scoring: {len(merged)}")

nli_engine = DeBERTaNLIEngine(
    model_id=config["models"]["utilization"]["model_id"],
    use_lora=config["models"]["utilization"]["use_lora"]
)
scorer = UtilizationScorer(nli_engine)

out_dir = Path(config["paths"]["utilization_dir"])
shard_mgr_su = ShardManager(out_dir, shard_size=config["execution"]["shard_size"])
items = merged.to_dict(orient="records")
remaining = shard_mgr_su.get_remaining_items(items, id_key="id")

shard_records = []
shard_idx = 0

for row in remaining:
    claims = list(row["claims"])
    passages = list(row["retrieved_passages"])
    
    s_u_cross_modal = scorer.compute_s_u(claims, passages, visual_context="scene details", is_text_only=False)
    s_u_text_only = scorer.compute_s_u(claims, passages, visual_context=None, is_text_only=True)
    
    shard_records.append({
        "id": row["id"],
        "dataset": row["dataset"],
        "s_u": s_u_cross_modal,
        "s_u_text_only": s_u_text_only,
        "is_hallucination_gt": row["is_hallucination_gt"]
    })
    
    if len(shard_records) >= config["execution"]["shard_size"]:
        shard_mgr_su.save_shard(shard_idx, shard_records, prefix="utilization_scores")
        shard_records = []
        shard_idx += 1

if shard_records:
    shard_mgr_su.save_shard(shard_idx, shard_records, prefix="utilization_scores")

save_run_metadata(out_dir, config, seed, "06_utilization", {"total_processed": len(items)})
print("\nStep 06 completed. Utilization scores saved.")
